In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

def show(name, value):
    print(f'{name}: {value}, shape={tuple(value.shape)}, dtype={value.dtype}')

In [2]:
def predict(x):
    # x 是一个 0 到 1 之间的浮点标量张量
    return torch.stack([1 - x, x])

x = torch.tensor(0.2)
y = torch.tensor([1., 0.])  # 真实情况：没有行人
y_pred = predict(x)

show('x', x)
show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)
show('预测概率之和', y_pred.sum())

x: 0.20000000298023224, shape=(), dtype=torch.float32
y（one-hot 真实标签）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32


In [3]:
scores = torch.log(y_pred)  # 从本例的概率构造等价 logits
label = y.argmax()        # [1, 0] → 类别编号 0，dtype 为 torch.int64

show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))
show('label（真实类别编号）', label)

loss = F.cross_entropy(scores, label)
show('loss', loss)

scores（传给损失函数的分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss: 0.2231435477733612, shape=(), dtype=torch.float32


In [4]:
manual_loss = -torch.log(y_pred[label])
show('手算：真实类别概率的负对数', manual_loss)
print('与 cross_entropy 一致：', torch.allclose(loss, manual_loss))

手算：真实类别概率的负对数: 0.2231435328722, shape=(), dtype=torch.float32
与 cross_entropy 一致： True


In [5]:
def compare(y, probabilities):
    label = y.argmax()
    # 仅为展示指定概率的交叉熵。log(0)=-inf，是这里刻意保留的边界情况。
    scores = torch.log(probabilities)
    loss = F.cross_entropy(scores, label)
    print(f'y={y.tolist()}, y_pred={probabilities.tolist()}')
    print(f'真实类别概率={probabilities[label].item():.4f}, loss={loss.item():.4f}')
    return loss

y = torch.tensor([1., 0.])
example_losses = []
for values in [[0.9, 0.1], [0.5, 0.5], [0.1, 0.9], [0., 1.], [1., 0.]]:
    example_losses.append(compare(y, torch.tensor(values)))
    print()

y=[1.0, 0.0], y_pred=[0.8999999761581421, 0.10000000149011612]
真实类别概率=0.9000, loss=0.1054

y=[1.0, 0.0], y_pred=[0.5, 0.5]
真实类别概率=0.5000, loss=0.6931

y=[1.0, 0.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.1000, loss=2.3026

y=[1.0, 0.0], y_pred=[0.0, 1.0]
真实类别概率=0.0000, loss=inf

y=[1.0, 0.0], y_pred=[1.0, 0.0]
真实类别概率=1.0000, loss=0.0000



In [6]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.1, 0.9])
compare(y, y_pred)

y=[0.0, 1.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.9000, loss=0.1054


tensor(0.1054)

In [7]:
scores = torch.tensor([2., -1.])
y = torch.tensor([1., 0.])
label = y.argmax()

show('原始分数 scores', scores)
show('给人看的预测概率', scores.softmax(dim=0))
loss = F.cross_entropy(scores, label)
show('直接调用 cross_entropy 得到的 loss', loss)

原始分数 scores: tensor([ 2., -1.]), shape=(2,), dtype=torch.float32
给人看的预测概率: tensor([0.9526, 0.0474]), shape=(2,), dtype=torch.float32
直接调用 cross_entropy 得到的 loss: 0.04858732968568802, shape=(), dtype=torch.float32


In [10]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.8, 0.2])

print(f"y shape: {tuple(y.shape)}, y_pred shape: {tuple(y_pred.shape)}")

label = y.argmax()

scores = torch.log(y_pred)
loss = F.cross_entropy(scores, label)

print(f"label: {label.item()}, loss: {loss.item():.4f}")

y shape: (2,), y_pred shape: (2,)
label: 1, loss: 1.6094


In [ ]:
修改 x 后的预测与损失： 修改前模型对真实类别的预测概率为0.8，损失较小（）； 修改后模型对真实类别的预测概率骤降
“有行人”实验的结果：真实类别为 1（“有行人”），模型预测该类别的概率仅为 ，计算得到的交叉熵损失与预期完全一致
两个极端例子的解释：[1, 0] 对 [0, 1] 的损失是无穷大；[1, 0] 对 [1, 0] 的损失是 0
概率与原始分数应该怎样传入损失函数：指定预测概率时 所以必须先对已知概率取对数，再送入函数才能得到正确的交叉熵；原始网络输出 scores 时 直接调用